## Exploring the CRCNS MT-2 dataset

**Imports**

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import imageio
import io
from IPython.display import display, HTML
import base64
import seaborn as sns
from scipy.signal import correlate

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Metal (MPS) backend:", device)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU:", device)
else:
    device = torch.device("cpu")
    print("Using CPU:", device)

from CRCNS_MT2_Dataset import CRCNS_MT2_Dataset

In [ ]:
data_path = "CRCNS_MT2"  

existing_neuron_indices = [
    1, 2, 5, 6, 8, 9, 10, 11, 13, 14,
    18, 21, 23, 24, 26, 27, 28, 30, 31,
    32, 37, 29, 40, 41, 42, 43, 47, 48,
    49, 50, 53, 55, 57, 58, 124, 125, 127,
    128, 120, 131, 132, 133, 137, 140, 141
]

**Dataset loading**

In [ ]:
chosen_indice = [8]

# Number of time steps per video clip (e.g., 2000 frames = 24s at 12ms/frame)
clip_length = 2000

# Spatial resolution to which all video clips will be resized (H, W)
spatial_resol = (128, 128)

# Whether to normalize video pixel values
normalize_videos = False

# Whether to normalize neuronal responses across time 
normalize_responses = False

# Whether to apply temporal smoothing to the neuronal responses
response_smoothing = False

# Whether to add artificial noise to the responses (e.g., for regularization)
add_noise = True

# Temporal resolution in milliseconds (sampling interval for both video and response)
temporal_resolution = 12

In [ ]:
dataset = CRCNS_MT2_Dataset(
    path=data_path,
    neuron_indices=chosen_indice,
    clip_length=clip_length,
    spatial_resol=spatial_resol,
    normalize_videos=normalize_videos, 
    normalize_responses=normalize_responses,
    response_smoothing=response_smoothing,
    add_noise=add_noise,
    temporal_resolution=temporal_resolution,
)

print("Dataset loaded successfully.")
print("Videos shape:", dataset.videos.shape)      # Expected shape (N, S, C, H, W, T)
print("Responses shape:", dataset.responses.shape)  # Expected shape (N, S, R, T)

Selecting the single virtual neuron to handle :

In [ ]:
neurons_to_handle = range(dataset.N_neurons)
print(f'We have {len(neurons_to_handle)} neurons to handle.')

dataset.select_population(neurons_to_handle)

**Dataset statistics**

In [ ]:
video_clips = dataset.videos[0]       # shape: (N_clips, 1, H, W, T)
response_clips = dataset.responses[0] # shape: (N_clips, 1, T)

N_clips, _, H, W, T = video_clips.shape
dt = dataset.dt  # ms

**Basics :**

In [ ]:
print(f"Number of neurons: {dataset.N_neurons}")
print(f"Number of video clips: {len(dataset)}")
print(f"Video shape: {dataset.videos.shape}")     # (1, N_clips, 1, H, W, T)
print(f"Response shape: {dataset.responses.shape}")  # (1, N_clips, 1, T)


**Clip length distribution (should be constant) :**

In [ ]:
clip_lengths = [clip.shape[-1] for clip in video_clips]  

plt.figure(figsize=(6, 4))
plt.hist(clip_lengths, bins=10, edgecolor='black')
plt.xlabel("Clip Length (frames)")
plt.ylabel("Count")
plt.title("Distribution of Clip Lengths")
plt.tight_layout()
plt.show()

**Average response amplitude per clip :**

In [ ]:
mean_responses = response_clips.mean(dim=-1).squeeze(1).numpy()  # shape: (N_clips,)
std_responses = response_clips.std(dim=-1).squeeze(1).numpy()

plt.figure(figsize=(8, 5))
plt.errorbar(np.arange(N_clips), mean_responses, yerr=std_responses, fmt='o')
plt.xlabel("Clip Index")
plt.ylabel("Mean Response ± STD")
plt.title("Response Amplitude Across Clips")
plt.grid(True)
plt.tight_layout()
plt.show()


**Heatmap of responses over time :**

In [ ]:
response_matrix = response_clips.squeeze(1).numpy()  # (N_clips, T)

white_red = LinearSegmentedColormap.from_list("white_red", ["white", "red"])

plt.figure(figsize=(12, 6), facecolor='white')
sns.heatmap(
    response_matrix,
    cmap=white_red,
    cbar_kws={'label': 'Firing Rate (a.u.)'},
    linewidths=0,
    linecolor=None
)

plt.xlabel("Time (frames)")
plt.ylabel("Clip Index")
plt.title("Raster-like Neural Response Heatmap", fontsize=14)
plt.tight_layout()
plt.show()

**Response autocorrelation :**

In [ ]:
clip_id = 20

signal = response_clips[clip_id, 0].numpy()
acf = correlate(signal - signal.mean(), signal - signal.mean(), mode='full')
acf = acf[acf.size // 2:]  # keep positive lags

plt.figure(figsize=(8, 4))
plt.plot(acf)
plt.title(f"Autocorrelation of Neural Response (Clip {clip_id})")
plt.xlabel("Lag (frames)")
plt.ylabel("Autocorrelation")
plt.grid(True)
plt.tight_layout()
plt.show()

**Frame brightness and response :**

In [ ]:
clip_id = 20

video = video_clips[clip_id, 0].numpy()  # (H, W, T)
frame_mean = video.mean(axis=(0, 1))     # (T,)
response = response_clips[clip_id, 0].numpy()

fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.plot(frame_mean, label="Frame Brightness", color='orange')
ax1.set_ylabel("Brightness", color='orange')
ax1.tick_params(axis='y', labelcolor='orange')

ax2 = ax1.twinx()
ax2.plot(response, label="Neural Response", color='lightblue')
ax2.set_ylabel("Neural Response", color='lightblue')
ax2.tick_params(axis='y', labelcolor='lightblue')

plt.title(f"Stimulus vs Response (Clip {clip_id})")
plt.xlabel("Time (frames)")
fig.tight_layout()
plt.show()


**Visualizing videos**

We choose one video to visualize :

In [ ]:
print(f'We have {dataset.S} videos to choose from.')

studied_video = 13
sample_video, sample_response, sample_signal_power, sample_nrn_mask = dataset[studied_video]

**Visualizing frames and a GIF from the sample :**

In [ ]:
number_of_frames_to_display = 10

# Ensure shape is (C=1, H, W, T), then convert to (H, W, T)
assert sample_video.ndim == 5, "Expected shape (1, 1, H, W, T)"
video_np = sample_video.squeeze().numpy()  # Shape: (H, W, T)

H, W, T = video_np.shape

**Frames :**

In [ ]:
frame_indices = np.linspace(0, T - 1, number_of_frames_to_display, dtype=int)

fig, axes = plt.subplots(1, number_of_frames_to_display, figsize=(20, 4))
for i, t in enumerate(frame_indices):
    ax = axes[i]
    ax.imshow(video_np[:, :, t], cmap='gray')
    ax.set_title(f"t={t}")
    ax.axis('off')
fig.suptitle("Sample Video (10 Frames)", fontsize=16)
plt.tight_layout()
plt.show()

**GIF :**

In [ ]:
gif_frames = [(255 * video_np[:, :, t]).astype(np.uint8) for t in range(T)]

gif_bytes = io.BytesIO()
imageio.mimsave(gif_bytes, gif_frames, format='GIF', duration=dataset.dt / 1000)
gif_bytes.seek(0)

gif_base64 = base64.b64encode(gif_bytes.read()).decode('utf-8')

display(HTML(f'<img src="data:image/gif;base64,{gif_base64}" width="258">'))

**Visualizing responses**

In [ ]:
print(f'We have {dataset.S} videos to choose from.')

studied_video = 12
sample_video, sample_response, sample_signal_power, sample_nrn_mask = dataset[studied_video]

Visualize the response for the chosen sample :

In [ ]:
response = sample_response[0, 0, :].numpy()  # shape: (T,)
t_axis = np.arange(response.shape[0])

plt.figure(figsize=(10, 4))
plt.plot(t_axis, response, color='blue')
plt.xlabel("Time (frames)")
plt.ylabel("Response (a.u.)")
plt.title("Neural Response to a Single Video Clip")
plt.grid(True)
plt.tight_layout()
plt.show()

**Visualizing signal power**

In [ ]:
print(f'We have {dataset.S} videos to choose from.')

studied_video = 10
sample_video, sample_response, sample_signal_power, sample_nrn_mask = dataset[studied_video]

Here, as all videos have been shown inly once, the signal power is $1$ everywhere.

In [ ]:
sp_np = sample_signal_power.numpy()
neuron_indices = np.arange(sp_np.shape[0])

sp_values = sp_np[:, 0] if sp_np.ndim > 1 else sp_np
plt.figure(figsize=(10, 4))
plt.bar(neuron_indices, sp_values)
plt.xlabel("Neuron Index")
plt.ylabel("Signal Power")
plt.title("Signal Power for Selected Neurons")
plt.tight_layout()
plt.show()